In [4]:
import numpy as np
from sklearn.model_selection import train_test_split

# 1. D님이 넘겨준 npy 파일 로드
X_raw = np.load('X.npy')
y = np.load('y.npy')

print(f"전체 데이터 형태: X={X_raw.shape}, y={y.shape}")

# 2. Train / Test 분할 (8:2 비율, 클래스 비율 유지)
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

전체 데이터 형태: X=(2827876, 70), y=(2827876,)


In [5]:
from sklearn.preprocessing import RobustScaler

# 네트워크 데이터의 극단적인 이상치(Outlier)에 강한 RobustScaler 사용
scaler = RobustScaler()

# X_train에만 fit(기준 설정)과 transform(변환)을 동시에 적용
X_train_scaled = scaler.fit_transform(X_train)

# X_test는 절대 fit하지 않고, Train의 기준에 맞춰 transform(변환)만 적용!
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# 1. 평가할 모델들 딕셔너리로 정의
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
    # "SVM": SVC(kernel='rbf', probability=True, random_state=42) # 매우 오래 걸릴 수 있으니 유의!
}

# 2. 성능 비교를 담을 리스트
performance_records = []

# 3. 모델별 학습 및 기본 평가 루프
for name, model in models.items():
    print(f"\n[{name}] 학습 시작...")
    model.fit(X_train_scaled, y_train) # 학습
    
    y_pred = model.predict(X_test_scaled) # 테스트 예측
    
    # 평가지표 계산
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    # 혼동 행렬 추출 (0: 정상, 1: 공격 이라고 가정)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    fpr = fp / (fp + tn) # 오탐률(False Positive Rate) 계산
    
    performance_records.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1,
        "False Positive Rate (FPR)": fpr
    })
    print(f"{name} 완료! 오탐률(FPR): {fpr*100:.4f}%")





[Logistic Regression] 학습 시작...


c:\python_project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression 완료! 오탐률(FPR): 10.1188%

[Random Forest] 학습 시작...
Random Forest 완료! 오탐률(FPR): 0.0658%

[Gradient Boosting] 학습 시작...
Gradient Boosting 완료! 오탐률(FPR): 0.1917%


KeyError: 'FPR'

In [10]:
!pip install tabulate


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
import pandas as pd

# 성능 비교 DataFrame 생성
df_performance = pd.DataFrame(performance_records)

# 💡 'FPR' 대신 딕셔너리에 넣었던 정확한 컬럼명인 'False Positive Rate (FPR)'로 수정합니다.
df_performance_sorted = df_performance.sort_values(by=['False Positive Rate (FPR)', 'F1-Score'], ascending=[True, False])

print("\n[모델별 성능 비교표]")
print(df_performance_sorted.to_markdown(index=False))


[모델별 성능 비교표]
| Model               |   Accuracy |   Precision |   Recall |   F1-Score |   False Positive Rate (FPR) |
|:--------------------|-----------:|------------:|---------:|-----------:|----------------------------:|
| Random Forest       |   0.99895  |    0.99895  | 0.99895  |   0.99895  |                 0.000658206 |
| Gradient Boosting   |   0.996011 |    0.996007 | 0.996011 |   0.996008 |                 0.00191738  |
| Logistic Regression |   0.810646 |    0.801321 | 0.810646 |   0.805281 |                 0.101188    |


In [30]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
import numpy as np

# 1. 아까 가장 성능이 좋았던 최종 모델 객체 가져오기
final_model = models["Random Forest"]

# 2. 5겹(5-Fold) 교차 검증 설정 (클래스 비율 유지)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Random Forest 교차 검증 시작")

# 3. 교차 검증 수행 (평가지표: F1-Score)
# 참고: X_train_scaled와 y_train 전체를 넣어 5번 쪼개가며 검증
cv_scores = cross_val_score(final_model, X_train_scaled, y_train, cv=cv, scoring='f1_weighted', n_jobs=-1)

print("\n[교차 검증 결과]")
print(f"각 Fold별 F1-Score: {np.round(cv_scores, 4)}")
print(f"평균 F1-Score: {cv_scores.mean():.4f} (± {cv_scores.std():.4f})")

Random Forest 교차 검증 시작

[교차 검증 결과]
각 Fold별 F1-Score: [0.9989 0.9988 0.9988 0.9989 0.9988]
평균 F1-Score: 0.9988 (± 0.0000)


In [34]:
# Random Forest를 최종 모델로 선정
best_model = models["Random Forest"]

# 기본 예측(predict) 대신 예측 확률(predict_proba)을 가져옴
y_pred_prob = best_model.predict_proba(X_test_scaled)[:, 1] # 클래스 1(공격)일 확률

# 기본 임계값은 0.5 (50%만 넘어도 공격으로 간주)
# 오탐을 줄이기 위해 '매우 확실할 때만(예: 85% 이상)' 공격으로 간주하도록 깐깐하게 변경
CUSTOM_THRESHOLD = 0.85 

# 새로운 임계값 적용
y_pred_strict = (y_pred_prob >= CUSTOM_THRESHOLD).astype(int)

# 깐깐해진 예측 결과로 다시 혼동 행렬 확인
new_cm = confusion_matrix(y_test, y_pred_strict)
tn, fp, fn, tp = new_cm.ravel()
new_fpr = fp / (fp + tn)

print(f"\n[임계값 {CUSTOM_THRESHOLD} 적용 후]")
print(f"오탐(False Positive) 건수: {fp}건")
print(f"새로운 오탐률(FPR): {new_fpr*100:.4f}%")


[임계값 0.85 적용 후]
오탐(False Positive) 건수: 229건
새로운 오탐률(FPR): 0.0504%


In [35]:
import joblib

# 1. 가장 성능이 좋은 Random Forest를 최종 모델로 선택
# 윤호 님이 백엔드 서버에서 모델을 재학습할 필요 없이 이 파일만 불러와서 실시간 탐지에 사용함
final_best_model = models["Random Forest"]

# 2. 모델과 스케일러를 파일로 저장
joblib.dump(final_best_model, 'ids_rf_model.joblib')
joblib.dump(scaler, 'ids_robust_scaler.joblib')

print("최종 모델 및 스케일러 저장 완료!")

최종 모델 및 스케일러 저장 완료!


In [ ]:
import joblib
import numpy as np
import uuid
from datetime import datetime

def analyze_realtime_traffic(traffic_features, metadata):
    """
    실시간 트래픽 특징 배열과 메타데이터를 받아 정해진 Alert JSON 형태로 반환합니다.
    - traffic_features: 1차원 리스트 또는 numpy 배열 
    - metadata: 원본 패킷의 IP, Port, 프로토콜 등을 담은 딕셔너리 (파이프라인 파트에서 전달)
    """
    
    # 1. 스케일러 및 모델 로드 (실제 서버에서는 전역 변수로 분리하여 로드 시간을 단축하세요)
    loaded_scaler = joblib.load('ids_robust_scaler.joblib')
    loaded_model = joblib.load('ids_rf_model.joblib')
    
    # 2. 데이터 스케일링
    input_array = np.array(traffic_features).reshape(1, -1)
    scaled_data = loaded_scaler.transform(input_array)
    
    # 3. 모델 추론
    prob = loaded_model.predict_proba(scaled_data)[0]
    attack_prob = float(prob[1]) # 1번 클래스(공격) 확률
    
    # 4. 임계값(Threshold) 확인
    is_threat = bool(attack_prob >= 0.80)
    
    # 5. 확률에 따른 동적 Risk Level 산정
    if attack_prob >= 0.95:
        risk_level = "CRITICAL"
    elif attack_prob >= 0.85:
        risk_level = "HIGH"
    elif attack_prob >= 0.80:
        risk_level = "MEDIUM"
    else:
        risk_level = "LOW"
        
    # 6. 요구사항에 맞춘 최종 Alert Event JSON (Case 2: model 기반) 생성
    return {
        # uuid를 활용해 고유한 model- ID 생성
        "alert_id": f"model-{uuid.uuid4().hex[:8]}", 
        # 현재 시간 자동 기록
        "timestamp": datetime.now().strftime("%Y-%m-%dT%H:%M:%S"), 
        
        # 메타데이터에서 추출 (없을 경우 기본값 반환)
        "src_ip": metadata.get("src_ip", "Unknown"),
        "dst_ip": metadata.get("dst_ip", "Unknown"),
        "src_port": metadata.get("src_port", 0),
        "dst_port": metadata.get("dst_port", 0),
        "protocol": metadata.get("protocol", "Unknown"),
        
        "attack_type": "AI Detected Anomaly" if is_threat else "Normal",
        "risk_level": risk_level if is_threat else "NONE",
        
        # 소수점 둘째 자리 포맷팅
        "confidence": round(attack_prob, 2), 
        
        "detection_type": "Model",
        
        # 챗봇이 자연어로 읽을 수 있는 Description 생성
        "description": f"AI 모델 탐지: 비정상 트래픽 패턴 감지 (확률 {round(attack_prob * 100)}%)" if is_threat else "정상적인 트래픽 흐름"
    }

# ==========================================
# 추론 담당자(이우진 님) 및 백엔드(조윤호 님) 사용 예시
# ==========================================
if __name__ == "__main__":
    # 1. 파이프라인에서 추출된 78개의 수치형 특징 배열
    sample_traffic = X_test[0] 
    
    # 2. 파이프라인에서 원본 패킷(PCAP)으로부터 파싱한 메타데이터
    # 이우진 님/조예한 님이 실시간 처리 시 아래와 같은 딕셔너리를 함께 만들어 윤호님에게 넘김.
    # 원본 패킷에 적혀있던 IP/Port 정보(메타데이터)를 따로 챙겨서 결과 JSON에 합쳐주어야 함. 모델 학습 시에는 과적합 방지를 위해 빼버렸습니다.
    sample_metadata = {
        "src_ip": "10.0.0.99",
        "dst_ip": "192.168.0.10",
        "src_port": 44521,
        "dst_port": 8080,
        "protocol": "TCP"
    }
    
    # 3. 함수 실행
    alert_json = analyze_realtime_traffic(sample_traffic, sample_metadata)
    
    print("출력된 Alert Event JSON:")
    import json
    print(json.dumps(alert_json, indent=2, ensure_ascii=False))


    # 이상이 있으면 알려주세요

출력된 Alert Event JSON:
{
  "alert_id": "model-9495647e",
  "timestamp": "2026-07-14T09:52:15",
  "src_ip": "10.0.0.99",
  "dst_ip": "192.168.0.10",
  "src_port": 44521,
  "dst_port": 8080,
  "protocol": "TCP",
  "attack_type": "AI Detected Anomaly",
  "risk_level": "CRITICAL",
  "confidence": 1.0,
  "detection_type": "Model",
  "description": "AI 모델 탐지: 비정상 트래픽 패턴 감지 (확률 100%)"
}


In [1]:
import joblib

# 1. 파일 불러오기 (파일 경로를 정확히 맞춰주세요)
feature_columns = joblib.load('feature_columns.pkl')

# 2. 내용 확인하기
print(f"총 피처 개수: {len(feature_columns)}개")
print("\n[피처 이름 리스트 전체 보기]")
for i, col in enumerate(feature_columns):
    print(f"{i+1}. {col}")

총 피처 개수: 70개

[피처 이름 리스트 전체 보기]
1. Destination Port
2. Flow Duration
3. Total Fwd Packets
4. Total Backward Packets
5. Total Length of Fwd Packets
6. Total Length of Bwd Packets
7. Fwd Packet Length Max
8. Fwd Packet Length Min
9. Fwd Packet Length Mean
10. Fwd Packet Length Std
11. Bwd Packet Length Max
12. Bwd Packet Length Min
13. Bwd Packet Length Mean
14. Bwd Packet Length Std
15. Flow Bytes/s
16. Flow Packets/s
17. Flow IAT Mean
18. Flow IAT Std
19. Flow IAT Max
20. Flow IAT Min
21. Fwd IAT Total
22. Fwd IAT Mean
23. Fwd IAT Std
24. Fwd IAT Max
25. Fwd IAT Min
26. Bwd IAT Total
27. Bwd IAT Mean
28. Bwd IAT Std
29. Bwd IAT Max
30. Bwd IAT Min
31. Fwd PSH Flags
32. Fwd URG Flags
33. Fwd Header Length
34. Bwd Header Length
35. Fwd Packets/s
36. Bwd Packets/s
37. Min Packet Length
38. Max Packet Length
39. Packet Length Mean
40. Packet Length Std
41. Packet Length Variance
42. FIN Flag Count
43. SYN Flag Count
44. RST Flag Count
45. PSH Flag Count
46. ACK Flag Count
47. URG Flag Coun

In [2]:
import joblib
import pandas as pd

# 1. 저장해둔 모델과 피처 리스트 파일 로드
loaded_model = joblib.load('ids_rf_model.joblib')
feature_columns = joblib.load('feature_columns.pkl')

# 2. 모델에서 피처 중요도(가중치) 추출
importances = loaded_model.feature_importances_

# 3. 중요도를 백분율(%)로 변환하여 데이터프레임으로 묶기
df_importance = pd.DataFrame({
    'Feature': feature_columns,
    'Importance (%)': importances * 100
})

# 4. 내림차순 정렬 및 Top 10 추출
top10_features = df_importance.sort_values(by='Importance (%)', ascending=False).head(10)

print("[탐지 기여도 Top 10 피처]")
print(top10_features.to_markdown(index=False))

[탐지 기여도 Top 10 피처]
| Feature                     |   Importance (%) |
|:----------------------------|-----------------:|
| Packet Length Variance      |          6.33978 |
| Bwd Packet Length Std       |          6.226   |
| Bwd Packet Length Mean      |          6.1391  |
| Average Packet Size         |          5.78557 |
| Destination Port            |          5.57903 |
| Avg Bwd Segment Size        |          4.5844  |
| Max Packet Length           |          3.90748 |
| Total Length of Bwd Packets |          3.79841 |
| Packet Length Mean          |          3.69502 |
| Packet Length Std           |          3.26538 |


In [ ]:
import numpy as np

# 예측 확률 및 깐깐한 임계값(0.8) 적용
y_pred_prob = loaded_model.predict_proba(X_test_scaled)[:, 1]
y_pred_strict = (y_pred_prob >= 0.80).astype(int)

# 오탐(False Positive) 인덱스 찾기: 실제는 0(정상)인데, 예측은 1(공격)인 조건
fp_indices = np.where((y_test == 0) & (y_pred_strict == 1))[0]

print(f"전체 테스트 데이터 중 오탐(FP) 건수: {len(fp_indices)}건")

# 오답 데이터 관찰하기
if len(fp_indices) > 0:
    # 스케일링 전의 '원본 수치'를 봐야 사람이 이해할 수 있으므로 X_test 사용
    df_fp = pd.DataFrame(X_test[fp_indices], columns=feature_columns)
    
    # 위에서 구한 Top 10 피처의 이름만 리스트로 추출
    top10_feature_names = top10_features['Feature'].tolist()
    
    print("\n[오탐 데이터의 원본 수치 (Top 10 핵심 피처만 보기)]")
    # 오탐 데이터가 많을 수 있으니 상위 5개만 샘플로 출력
    print(df_fp[top10_feature_names].head(5).to_markdown())
else:
    print("오탐 없")

전체 테스트 데이터 중 오탐(FP) 건수: 233건

[오탐 데이터의 원본 수치 (Top 10 핵심 피처만 보기)]
|    |   Packet Length Variance |   Bwd Packet Length Std |   Bwd Packet Length Mean |   Average Packet Size |   Destination Port |   Avg Bwd Segment Size |   Max Packet Length |   Total Length of Bwd Packets |   Packet Length Mean |   Packet Length Std |
|---:|-------------------------:|------------------------:|-------------------------:|----------------------:|-------------------:|-----------------------:|--------------------:|------------------------------:|---------------------:|--------------------:|
|  0 |                 12       |                       0 |                        6 |                     3 |                443 |                      6 |                   6 |                             6 |              2       |              3.4641 |
|  1 |                  5.33333 |                       0 |                        6 |                     5 |              28201 |                      6 |           